# Project Jaundice — Kaggle training runbook v2

Ready-to-run driver. It **clones the repo** (first cell), builds the manifest with the Kaggle
dataset path, trains the ERM baseline + the method, then runs the counterfactual / fairness /
retrieval evaluations.

**Before Run-All:** Settings -> Accelerator = **GPU T4 x2**, Internet = **ON**, and add your
uploaded `jaundicedataset3` dataset via **Add Input** (right panel).

Metrics land in `experiments/<tag>/metrics.json`: `test` (@0.5), `test_youden`, `test_screening`
(recall-first), threshold-free `auc`, and `test_fairness` (per-skin-tone-stratum + worst-group gaps).


## 1 - Get the code (clone, or pull if already cloned)

In [ ]:
import os
REPO = "https://github.com/bad-eastwind/projectjaundicce.git"
%cd /kaggle/working
if os.path.isdir("projectjaundicce/.git"):
    %cd projectjaundicce
    !git pull --ff-only
else:
    !git clone $REPO
    %cd projectjaundicce


## 2 - Environment + deps

In [ ]:
%env PYTHONPATH=src
!pip -q install -U timm
# Kaggle has ~4 CPU cores: 8 dataloader workers oversubscribes -> warning / freeze risk on long runs.
!sed -i 's/num_workers: 8/num_workers: 2/' configs/base.yaml
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-ONLY (set Accelerator=GPU)")


## 3 - Locate the dataset + point the config at it

Auto-detects the folder under `/kaggle/input` that contains `jaundice/` + `normal/` subdirs and writes
that path into `configs/kaggle.yaml`. If nothing is found, add the dataset via **Add Input**.

In [ ]:
import os, yaml
cands = []
for root, dirs, _ in os.walk("/kaggle/input"):
    dl = set(x.lower() for x in dirs)
    if any("jaundice" in x for x in dl) and any("normal" in x for x in dl):
        cands.append(root)
assert cands, "Dataset not found under /kaggle/input. Add your jaundicedataset3 via Add Input."
DATA_ROOT = cands[0]
print("DATA_ROOT =", DATA_ROOT)
cfg = yaml.safe_load(open("configs/kaggle.yaml"))
cfg.setdefault("data", {})["neo_skin_core"] = DATA_ROOT
yaml.safe_dump(cfg, open("configs/kaggle.yaml", "w"))
print("patched configs/kaggle.yaml -> neo_skin_core =", DATA_ROOT)


## 4 - Build manifest + discover pseudo-domains (first values)

`manifest` writes `data/manifest.csv` (all downstream steps read image paths from here, so the dataset
path only matters at this step). `domains` prints per-domain counts, positive-rate, and the
**label-shift spread** diagnostic (flags when LOCO conflates label shift with domain shift).

In [ ]:
!python -m jaundice.data.manifest --config configs/kaggle.yaml
!python -m jaundice.data.domains  --config configs/kaggle.yaml --k 3


## 5 - Metrics reader

In [ ]:
import json, glob
def show(tag):
    ps = sorted(glob.glob(f"experiments/{tag}-*/metrics.json"))
    if not ps:
        print(tag, "-> no metrics yet"); return None
    d = json.load(open(ps[-1]))
    r = lambda x: {k: round(v, 3) for k, v in (x or {}).items()}
    print("==", tag, "==")
    print("  test@0.5   :", r(d.get("test")))
    print("  test@youden:", r(d.get("test_youden")))
    print("  screening  :", r(d.get("test_screening")))
    print("  thresholds :", d.get("thresholds"))
    f = d.get("test_fairness") or {}
    print("  fairness gap bacc/sens:",
          round(f.get("bacc_gap", float("nan")), 3), round(f.get("sensitivity_gap", float("nan")), 3))
    return d


## 6 - GPU smoke (fast pipeline sanity; not a real run)

In [ ]:
!python -m jaundice.train.train --config configs/smoke.yaml --tag smoke


## 7 - Early insurance: ERM baseline (finishes first)

Trained first so that even if the long grid below times out, you already have the naive baseline +
the checkpoint the counterfactual test needs. `scin=False physics=none causal=off`; a ~0.98
in-distribution AUC here is the *lighting shortcut*, not a win — the separation shows up under LOCO,
counterfactuals, and the fairness gap.

In [ ]:
!python -m jaundice.train.train --config configs/erm.yaml --tag erm_indist
show("erm_indist")


## 8 - Early insurance: headline model (BiliGrad, full stack)

`ourbg_indist` = SCIN + MixStyle + causal adversaries + cephalocaudal differential bilirubin field.
Trained early so the counterfactual comparison + a headline checkpoint exist before the full sweep.

In [ ]:
!python -m jaundice.train.train --config configs/bili_grad.yaml --tag ourbg_indist
show("ourbg_indist")


## 9 - Counterfactual causal-validity + retrieval explanations (inference on the two checkpoints)

Nuisance interventions (illuminant; melanin with b* held fixed) on test images. Ours should show low
`pred_swing`/`beta_cov` while ERM swings, especially on the melanin intervention; `raw_b*_swing` large
under illuminant proves the intervention is real. (You can also re-run all of this on your Mac later.)

In [ ]:
import glob
OURS = sorted(glob.glob("experiments/ourbg_indist-*"))[-1]
ERM  = sorted(glob.glob("experiments/erm_indist-*"))[-1]
!python -m jaundice.eval.counterfactual --config configs/bili_grad.yaml --ckpt $OURS/best.pt --num 64 \
    --baseline-config configs/erm.yaml --baseline-ckpt $ERM/best.pt
!python -m jaundice.explain.run --config configs/bili_grad.yaml --ckpt $OURS/best.pt --num-queries 20
print(open(sorted(glob.glob("experiments/cf-*/counterfactual.md"))[-1]).read())


## 10 - COMPLETE TRAINING GRID (the one-shot: everything the paper needs)

Trains every config once, plus error bars, **ordered by priority** so the essential tables finish
first (each run early-stops on val AUC, so ~4-6 min each):

1. **Main comparison + LOCO** (seed 42): erm / irm / groupdro / ours / ours_bg x {indist, hold-0, hold-1, hold-2} = 20 runs
2. **Ablations** (in-distribution, seed 42): -scin / -disentangle / -causal / -mixstyle / -lora = 6 runs
3. **Error bars**: repeat the main comparison under 2 more seeds = 40 runs

~66 trainings, ~5-7 h total (within Kaggle's 12 h commit limit). Every run saves
`experiments/<tag>/best.pt` + `metrics.json`. If it times out mid-way, steps 1-2 (the point
estimates) are already done and saved. Each sweep writes incremental `results.csv`/`results.md`.

In [ ]:
# 1) main comparison + LOCO (seed 42)
!python -m jaundice.train.sweep --matrix method   --holdouts indist,0,1,2
# 2) ablations, in-distribution (seed 42)
!python -m jaundice.train.sweep --matrix ablation --holdouts indist
# 3) error bars: 2 more seeds on the main comparison + LOCO
!python -m jaundice.train.sweep --matrix method   --holdouts indist,0,1,2 --seeds 1,2
import glob
for r in sorted(glob.glob("experiments/sweep-*/results.md")):
    print("\n=====", r, "=====\n"); print(open(r).read())


## 11 - Results summary + weights inventory (what to download)

Aggregates every run's `metrics.json` into one table (`/kaggle/working/RESULTS.{md,csv}`) and lists all
`best.pt` checkpoints with sizes. `/kaggle/working` is saved on **Save Version (Commit)**.

**To pull to your Mac:** Output tab -> download `RESULTS.md/csv`, `experiments/*/metrics.json`, the
`best.pt` you want, `counterfactual.md`, `explanations.md`. Each `best.pt` embeds its own `cfg`, so
locally you rebuild the manifest (pointing at your local dataset) and load the checkpoint directly.
If CUDA OOM at a train step: `!sed -i 's/batch_size: 32/batch_size: 16/' configs/base.yaml`.

In [ ]:
import json, glob, os
rows = []
for mj in sorted(glob.glob("experiments/*/metrics.json")):
    d = json.load(open(mj))
    t, ty, f = d.get("test") or {}, d.get("test_youden") or {}, d.get("test_fairness") or {}
    g = lambda x, k: round(x.get(k, float("nan")), 3)
    rows.append({"run": os.path.basename(os.path.dirname(mj)), "auc": g(t, "auc"),
                 "bacc@youden": g(ty, "balanced_acc"), "sens@youden": g(ty, "sensitivity"),
                 "spec@youden": g(ty, "specificity"), "fair_gap_bacc": g(f, "bacc_gap")})
try:
    import pandas as pd
    df = pd.DataFrame(rows); print(df.to_string(index=False))
    df.to_csv("/kaggle/working/RESULTS.csv", index=False)
    open("/kaggle/working/RESULTS.md", "w").write("# Results\n\n" + df.to_string(index=False) + "\n")
except Exception:
    print(json.dumps(rows, indent=2)); open("/kaggle/working/RESULTS.md", "w").write(json.dumps(rows, indent=2))

ckpts = sorted(glob.glob("experiments/*/best.pt"))
tot = sum(os.path.getsize(c) for c in ckpts) / 1e6
print(f"\n{len(rows)} runs | {len(ckpts)} checkpoints | {tot:.0f} MB total weights")
print("saved: /kaggle/working/RESULTS.md, RESULTS.csv")
